In [ ]:
# get bvtrain (shared training plumbing): locally it's ../bvtrain; on colab/kaggle we clone the repo
import os, sys
_CANDS = ["..", ".", "botanical-vision"]
if not any(os.path.isdir(f"{p}/bvtrain") for p in _CANDS):
    os.system("git clone -q https://github.com/babnigg/botanical-vision.git")
for _p in _CANDS:
    if os.path.isdir(f"{_p}/bvtrain"):
        sys.path.insert(0, _p)
        break

import pandas as pd
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
import numpy as np
import bvtrain as bv

import os
os.makedirs("checkpoints", exist_ok=True)

env = bv.setup()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 32
EPOCHS = 5
LR = 3e-5
N_SPECIES = None   # int (e.g. 100) for a quick subset run, None for all 4,094 species

data = bv.load_data(env, n_species=N_SPECIES)
hf = data._hf
if hf is None:                     # local-files machine: pull the HF build directly
    hf = load_dataset(env.hf_repo)

labels = data.labels
num_labels = data.n_labels
print(f"{num_labels} species | train={len(hf['train'])} val={len(hf['val'])} test={len(hf['test'])}")

In [ ]:
#preprocecssing
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

In [ ]:
# picklable transform wrapper (fixes the num_workers>0 pickle error from a nested function)
class TransformFn:
    def __init__(self, tfm):
        self.tfm = tfm
    def __call__(self, batch):
        batch["pixel_values"] = [self.tfm(img.convert("RGB")) for img in batch["image"]]
        return batch

train_ds = hf["train"].with_transform(TransformFn(train_transform))
val_ds   = hf["val"].with_transform(TransformFn(eval_transform))
test_ds  = hf["test"].with_transform(TransformFn(eval_transform))

def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    labels_t = torch.tensor([b["label"] for b in batch])
    return {"pixel_values": pixel_values, "labels": labels_t}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

In [ ]:
#model
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)
model.to(device)

# resume across kaggle sessions: mount the previous run's output as an input
# (kernel_sources in kernel-metadata.json) and continue from its best checkpoint
import glob
START_EPOCH = 0
_prev = glob.glob("/kaggle/input/**/vit_species_best.pt", recursive=True)
if _prev:
    model.load_state_dict(torch.load(_prev[0], map_location=device))
    START_EPOCH = 3          # epochs completed in the capped session
    print(f"resumed from {_prev[0]}, continuing at epoch {START_EPOCH + 1}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
#training/eval

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_t = batch["labels"].to(device)
            outputs = model(pixel_values=pixel_values).logits
            loss = criterion(outputs, labels_t)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * pixel_values.size(0)
            correct += (outputs.argmax(-1) == labels_t).sum().item()
            total += pixel_values.size(0)
    return total_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(START_EPOCH, EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "checkpoints/vit_species_best.pt")
        print("  -> saved new best checkpoint")

In [ ]:
def evaluate_topk(loader, k=5):
    model.eval()
    top1, top5, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_t = batch["labels"].to(device)
            outputs = model(pixel_values=pixel_values).logits
            _, topk_preds = outputs.topk(k, dim=-1)

            top1 += (topk_preds[:, 0] == labels_t).sum().item()
            top5 += (topk_preds == labels_t.unsqueeze(1)).any(dim=1).sum().item()
            total += labels_t.size(0)
    return top1 / total, top5 / total

model.load_state_dict(torch.load("checkpoints/vit_species_best.pt"))
top1, top5 = evaluate_topk(test_loader, k=5)
print(f"Test top-1: {top1:.3f} | top-5: {top5:.3f}")